In [ ]:
# Cell to Clear Memory
import torch
import gc

# Delete any existing model or large tensor variables
if 'model' in locals():
    del model
if 'tokenizer' in locals():
    del tokenizer
if 'inputs' in locals():
    del inputs

# Force Python's garbage collector to run
gc.collect()

# Clear the PyTorch CUDA cache (most effective step)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared.")
else:
    print("Not using CUDA, moving on.")

CUDA cache cleared.


In [ ]:
# Cell 1: Setup and Library Imports

# Install/update necessary libraries (run this first in a new session)
!pip install --upgrade transformers sentencepiece accelerate pandas numpy

# Import standard libraries
import os
import pandas as pd
import numpy as np

# Import PyTorch and Hugging Face libraries
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
# Explicit T5 Tokenizer for stability with SentencePiece
from transformers.models.t5.tokenization_t5 import T5Tokenizer
from google.colab import drive

print("Setup and imports complete.")

Setup and imports complete.


In [ ]:
# Configuration and Data Loading

#Configuration Settings
MODEL_NAME = "google/metricx-24-hybrid-xl-v2p6"
TOKENIZER_BASE_MODEL = "google/mt5-xl"
BATCH_SIZE = 2 # Adjustable batch size for GPU/CPU memory

# Google Drive and File Setup
drive.mount('/content/drive')
FILE_PATH = '/content/drive/MyDrive/AI351ProjectTest/02-TranslationMetrics/'

sub_samplename = 'subsample10_mt.csv'

FILE_PATH2 = os.path.join(FILE_PATH, sub_samplename)
SELECTED_COLUMNS2 = ["text", "label", "label_desc", "translation"]

# Read the File into a DataFrame
try:
    df_sub = pd.read_csv(
        FILE_PATH2,
        usecols=SELECTED_COLUMNS2,
        encoding="latin1"
    )
    df = df_sub.copy() # Create a working copy

    print(f"DataFrame loaded successfully with {len(df)} rows.")
    print(f"\nFirst 5 rows of the working DataFrame (df):")
    display(df.head())

except FileNotFoundError:
    print(f"ERROR: File not found at the specified path: {FILE_PATH2}")
    df = pd.DataFrame() # Create an empty DataFrame to avoid errors later
except ValueError:
    print(f"ERROR: Check the column names. They might not exist in the file: {SELECTED_COLUMNS2}")
    df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    df = pd.DataFrame()

Mounted at /content/drive
DataFrame loaded successfully with 3783 rows.

First 5 rows of the working DataFrame (df):


,text,label,label_desc,translation
0,mental health using social anxiety and depress...,0,sadness,Kalusugan ng isip gamit ang sosyal na pag-aala...
1,sleepless night become more frequent a i slip ...,0,sadness,Ang mga gabi nang walang tulog ay lalong madal...
2,man utd star paul pogba open up on depression ...,0,sadness,Ibinahagi ni Manchester United star na si Paul...
3,man utd star paul pogba open up on depression ...,0,sadness,Ibinahagi ni Manchester United star na si Paul...
4,look like the first stage of depression for mo...,0,sadness,Parang unang yugto ng depresyon para sa karami...


In [ ]:
# Model and Scoring Helper Functions

def load_metricx24_model():
    """Loads the MetricX-24 model and tokenizer with the necessary regression config."""
    # (Function body remains the same as before)
    try:
        # Load Tokenizer (using T5Tokenizer for stability)
        tokenizer = T5Tokenizer.from_pretrained(
            TOKENIZER_BASE_MODEL,
            use_fast=False
        )

        # Load Configuration and set num_labels=1 for single-value regression output
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=1)

        # Load the Model
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(device)

        print(f"\n✅ Model loaded successfully on device: {device}")
        return tokenizer, model, device

    except Exception as e:
        print(f"❌ ERROR: Failed to load model components: {e}")
        return None, None, None


def score_dataframe_qe(df, source_col, hypothesis_col, tokenizer, model, device, batch_size):
    """
    Processes a DataFrame in Reference-Free (QE) mode and returns the MetricX-24 RAW scores (0-25, L.I.B.),
    including a progress printout.
    """
    all_raw_scores = []

    if df.empty:
        print("Input DataFrame is empty. Skipping scoring.")
        return all_raw_scores

    total_samples = len(df)
    total_batches = (total_samples + batch_size - 1) // batch_size # Ceiling division

    # 1. Prepare Input Texts
    input_texts = [
        f"source: {src} hypothesis: {hyp} reference: "
        for src, hyp in zip(df[source_col], df[hypothesis_col])
    ]

    # 2. Process in Batches
    model.eval()
    print(f"\n🚀 Starting scoring for {total_samples} samples across {total_batches} batches...")

    for batch_num, i in enumerate(range(0, total_samples, batch_size), 1):

        # --- Progress Printout ---
        current_progress = (batch_num / total_batches) * 100
        print(f"Batch {batch_num}/{total_batches} processed ({current_progress:.1f}%)", end='\r')
        # -------------------------

        batch_texts = input_texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1536
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            batch_scores = outputs.logits.squeeze().cpu().tolist()

            if isinstance(batch_scores, float):
                batch_scores = [batch_scores]

            all_raw_scores.extend(batch_scores)

    # Print a final newline after the progress indicator
    print("\n✅ All batches completed.")
    return all_raw_scores

print("Helper functions defined.")

Helper functions defined.


In [ ]:
# Scoring, Scaling, and Display (Execution Block)

# --- Scaling Function ---
def scale_to_batayan(raw_scores):
    # (Function body remains the same as before)
    raw_scores = np.array(raw_scores)
    clipped_scores = np.clip(raw_scores, 0.0, 25.0)
    quality_scores = 100 * (1 - (clipped_scores / 25.0))
    return quality_scores.tolist()

# -------------------- MAIN EXECUTION --------------------
if not df.empty:

    # Load the Model
    tokenizer, model, device = load_metricx24_model()

    if model is not None:

        # Score the DataFrame to get RAW Scores (0-25, L.I.B.)
        raw_scores = score_dataframe_qe(
            df,
            source_col='text',
            hypothesis_col='translation',
            tokenizer=tokenizer,
            model=model,
            device=device,
            batch_size=BATCH_SIZE
        )

        df['MetricX24_Raw'] = raw_scores

        # Scale the Raw Scores to BATAYAN Standard (0-100, H.I.B.)
        df['MetricX24_Scaled_BATAYAN'] = scale_to_batayan(df['MetricX24_Raw'])

        # Display Results
        print("\n--- Scored Results ---")
        print("Interpretation: Raw (0-25, Lower is Better). Scaled (0-100, Higher is Better, aligned with BATAYAN)")
        display(df[['text', 'translation', 'MetricX24_Raw', 'MetricX24_Scaled_BATAYAN']].head(10))
    else:
        print("\nSkipping scoring due to model loading failure.")
else:
    print("\nSkipping scoring as DataFrame is empty.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

...

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of MT5ForSequenceClassification were not initialized from the model checkpoint at google/metricx-24-hybrid-xl-v2p6 and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ Model loaded successfully on device: cuda

🚀 Starting scoring for 3783 samples across 1892 batches...
Batch 1892/1892 processed (100.0%)
✅ All batches completed.

--- Scored Results ---
Interpretation: Raw (0-25, Lower is Better). Scaled (0-100, Higher is Better, aligned with BATAYAN)


,text,translation,MetricX24_Raw,MetricX24_Scaled_BATAYAN
0,mental health using social anxiety and depress...,Kalusugan ng isip gamit ang sosyal na pag-aala...,0.333612,98.665554
1,sleepless night become more frequent a i slip ...,Ang mga gabi nang walang tulog ay lalong madal...,0.324862,98.700554
2,man utd star paul pogba open up on depression ...,Ibinahagi ni Manchester United star na si Paul...,0.299292,98.802832
3,man utd star paul pogba open up on depression ...,Ibinahagi ni Manchester United star na si Paul...,0.307535,98.769858
4,look like the first stage of depression for mo...,Parang unang yugto ng depresyon para sa karami...,0.366435,98.534259
5,post concert depression except it s post hocke...,"Depresyon pagkatapos ng konsyerto, pero parang...",0.354770,98.580920
6,who launch yearlong campaign to fight depressi...,Sino ang naglulunsad ng taong kampanya laban s...,0.360805,98.556780
7,depression suck especially accompanied by inso...,"Ang depresyon ay nakakabigo, lalo na kung kasa...",0.295899,98.816405
8,do i play pjsekai to forget the fact i wa clin...,Dapat ba akong maglaro ng PJSEkai para kalimut...,0.351864,98.592544
9,mizzzidc imagine making sacrifice just to rais...,"Mizzzidc, isipin mo, nagsasakripisyo ka lang p...",0.368693,98.525226


In [ ]:
# Saving Results

if not df.empty and 'MetricX24_Scaled_BATAYAN' in df.columns:
    try:
        # Construct the output filename: e.g., subsample01_mt.csv -> subsample01_MTScore.csv
        base_name, ext = os.path.splitext(sub_samplename)
        output_filename = f"{base_name}_MTScore{ext}"
        OUTPUT_FILE_PATH = os.path.join(FILE_PATH, output_filename)

        # Save the DataFrame with the new scores
        df.to_csv(OUTPUT_FILE_PATH, index=False, encoding='utf-8')

        print(f"\n✅ Results successfully saved to: {OUTPUT_FILE_PATH}")
        print(f"File name used: {output_filename}")

    except Exception as e:
        print(f"\n❌ ERROR: Failed to save the file: {e}")
else:
    print("\nSkipping save step as the DataFrame is empty or scoring failed.")


✅ Results successfully saved to: /content/drive/MyDrive/AI351ProjectTest/02-TranslationMetrics/subsample10_mt_MTScore.csv
File name used: subsample10_mt_MTScore.csv


In [2]:
import pandas as pd
import os

# Reconstruct the file path for the saved CSV
FILE_PATH = '/content/drive/MyDrive/AI351ProjectTest/02-TranslationMetrics/'
output_filename = 'subsample10_mt_MTScore.csv'
IMPORT_FILE_PATH = os.path.join(FILE_PATH, output_filename)

try:
    df_imported = pd.read_csv(IMPORT_FILE_PATH)
    print(f"Successfully imported '{output_filename}' into a new DataFrame 'df_imported'.")
    print(f"DataFrame has {len(df_imported)} rows and {len(df_imported.columns)} columns.")
    print("First 5 rows of the imported DataFrame:")
    display(df_imported.head())
except FileNotFoundError:
    print(f"ERROR: The file '{output_filename}' was not found at '{IMPORT_FILE_PATH}'.")
except Exception as e:
    print(f"An error occurred while importing the file: {e}")

ERROR: The file 'subsample10_mt_MTScore.csv' was not found at '/content/drive/MyDrive/AI351ProjectTest/02-TranslationMetrics/subsample10_mt_MTScore.csv'.
